# ERCOT EMIL — Quick-Reference Snippet

Self-contained. Set `PRODUCT` and run all.

**Phase 1 — Learn:** unfiltered fetch (API defaults to today) to discover field names, data types, and example values.  
**Phase 2 — Extract:** fill `QUERY_PARAMS` using what Phase 1 revealed, then issue a fresh API call.

In [1]:
import os, time
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

_TOKEN_URL = (
    'https://ercotb2c.b2clogin.com'
    '/ercotb2c.onmicrosoft.com'
    '/B2C_1_PUBAPI-ROPC-FLOW'
    '/oauth2/v2.0/token'
)
_CLIENT_ID   = 'fec253ea-0d06-4272-a5e6-b478baeecd70'
_token_cache = {'token': None, 'expires_at': 0.0}

def _get_token() -> str:
    if _token_cache['token'] and time.time() < _token_cache['expires_at']:
        return _token_cache['token']
    resp = requests.post(_TOKEN_URL, data={
        'username'     : os.environ['ERCOT_USERNAME'],
        'password'     : os.environ['ERCOT_PASSWORD'],
        'grant_type'   : 'password',
        'client_id'    : _CLIENT_ID,
        'scope'        : f'openid {_CLIENT_ID} offline_access',
        'response_type': 'id_token',
    }, timeout=(10, 60))
    resp.raise_for_status()
    body = resp.json()
    _token_cache['token']      = body['id_token']
    _token_cache['expires_at'] = time.time() + int(body.get('expires_in', 3600)) - 60
    return _token_cache['token']

def _hdrs() -> dict:
    return {
        'Authorization'            : f'Bearer {_get_token()}',
        'Ocp-Apim-Subscription-Key': os.environ['ERCOT_SUBSCRIPTION_KEY'],
        'Accept'                   : 'application/json',
    }

def _get(url, **kwargs):
    r = requests.get(url, headers=_hdrs(), timeout=(10, 120), **kwargs)
    r.raise_for_status()
    return r.json()

def _to_df(body: dict) -> pd.DataFrame:
    cols = [f['name'] if isinstance(f, dict) else f for f in body.get('fields', [])]
    return pd.DataFrame([dict(zip(cols, row)) for row in body.get('data', [])])

print('Auth OK —', _get_token()[:20], '...')

Auth OK — eyJhbGciOiJSUzI1NiIs ...


In [2]:
# ── PARAMETERS ─────────────────────────────────────────────────────────────────
PRODUCT: str = 'NP6-788-CD'   # Any ERCOT EMIL product code
# ──────────────────────────────────────────────────────────────────────────────

---
## Phase 1 — Learn
No filter params — API defaults to today's window. Use the output to populate `QUERY_PARAMS` in Phase 2.

In [3]:
API_BASE = 'https://api.ercot.com/api/public-reports'

# discover sub-report from artifacts
product_meta = _get(f'{API_BASE}/{PRODUCT.lower()}')
artifacts    = product_meta.get('artifacts', [])
if not artifacts:
    raise RuntimeError(f'No artifacts found for {PRODUCT}.')
if len(artifacts) > 1:
    print('Multiple sub-reports — using first:')
    for a in artifacts:
        print(f'  {a["_links"]["endpoint"]["href"]}  —  {a["displayName"]}')

artifact = artifacts[0]
endpoint = artifact['_links']['endpoint']['href']

# schema: field names, types, which support range filters
schema    = _get(endpoint)
fields    = schema['fields']
fields_df = pd.DataFrame([{
    'name'      : f['name'],
    'label'     : f['label'],
    'dataType'  : f['dataType'],
    'searchable': f.get('searchable', False),
    'hasRange'  : f.get('hasRange', False),
} for f in fields])

print(f'Product  : {PRODUCT} — {artifact["displayName"]}')
print(f'Endpoint : {endpoint}')
print()
print('Fields (hasRange=True → supports XxxFrom / XxxTo query params):')
display(fields_df)

# unfiltered sample — no date params, API fills in today by default
sample_body = _get(endpoint, params={'page': 1, 'size': 5})
sample_meta = sample_body.get('_meta', {})
sample_df   = _to_df(sample_body)

print(f'Default window applied by API : {sample_meta.get("query", {}).get("parameters")}')
print(f'totalRecords in that window   : {sample_meta.get("totalRecords")}')
print()
print('Sample rows:')
display(sample_df)

Product  : NP6-788-CD — LMPs by Resource Nodes, Load Zones and Trading Hubs
Endpoint : https://api.ercot.com/api/public-reports/np6-788-cd/lmp_node_zone_hub

Fields (hasRange=True → supports XxxFrom / XxxTo query params):


,name,label,dataType,searchable,hasRange
0,SCEDTimestamp,SCED Time Stamp,DATETIME,True,True
1,repeatHourFlag,Repeat Hour Flag,BOOLEAN,True,False
2,settlementPoint,Settlement Point,VARCHAR,True,False
3,LMP,LMP,DOUBLE,True,True


Default window applied by API : {'SCEDTimestampFrom': '2026-05-28T00:00:00', 'SCEDTimestampTo': '2026-05-28T23:59:59'}
totalRecords in that window   : 203320

Sample rows:


,SCEDTimestamp,repeatHourFlag,settlementPoint,LMP
0,2026-05-28T15:10:19,False,7RNCHSLR_ALL,28.50
1,2026-05-28T15:10:19,False,A4_DGR1_RN,28.30
2,2026-05-28T15:10:19,False,A4_DGR2_RN,28.30
3,2026-05-28T15:10:19,False,ABINDUST_RN,33.26
4,2026-05-28T15:10:19,False,ADL_RN,57.39


---
## Phase 2 — Extract
Fill `QUERY_PARAMS` using the field names and value examples from Phase 1, then run the cell below to issue a fresh API call.

In [4]:
# ── EXTRACTION PARAMETERS (fill in after Phase 1) ──────────────────────────────
QUERY_PARAMS: dict = {
    'SCEDTimestampFrom': '2022-03-03T06:00:00', 
    'SCEDTimestampTo'  : '2025-11-11T15:00:00',
}
PAGE_SIZE: int = 1000   # ERCOT max is 1000
# ──────────────────────────────────────────────────────────────────────────────

In [5]:
# page 1 — get total page count first
body = _get(endpoint, params={**QUERY_PARAMS, 'page': 1, 'size': PAGE_SIZE})
meta        = body.get('_meta', {})
total_pages = meta.get('totalPages', 1)
total_recs  = meta.get('totalRecords', '?')

print(f'params       : {QUERY_PARAMS}')
print(f'totalRecords : {total_recs}')
print(f'totalPages   : {total_pages}  (page_size={PAGE_SIZE})')

# fetch remaining pages
all_dfs = [_to_df(body)]
for page in range(2, total_pages + 1):
    b = _get(endpoint, params={**QUERY_PARAMS, 'page': page, 'size': PAGE_SIZE})
    all_dfs.append(_to_df(b))
    if page % 10 == 0 or page == total_pages:
        print(f'  fetched page {page}/{total_pages} …')

df = pd.concat(all_dfs, ignore_index=True)
print(f'\ndone — {len(df):,} rows')
df.head(10)

params       : {'SCEDTimestampFrom': '2022-03-03T06:00:00', 'SCEDTimestampTo': '2025-11-11T15:00:00'}
totalRecords : 196640092
totalPages   : 196641  (page_size=1000)
  fetched page 10/196641 …
  fetched page 20/196641 …


KeyboardInterrupt: 